# LLM-Powered Agent for Pokémon TCG AI Battle Challenge
This notebook implements an agent for the Simulation Category that uses a pre-trained Large Language Model (LLM) to select the optimal move.

## ⚠️ KAGGLE SUBMISSION ERRORS FIXED ⚠️
1. **Offline Evaluation Error:** When you submit an agent to a Kaggle competition leaderboard, the evaluation servers do **not** have internet access. Calling `from_pretrained("Qwen...")` will crash. You MUST add the model to your notebook as a Kaggle Dataset and load it from `/kaggle/input/...`.
2. **Single File Requirement:** Kaggle environments require a single `submission.py` file. The code below now uses `%%writefile submission.py` to package your imports, model initialization, and the agent function into one self-contained file.
3. **CPU Evaluation Limits:** Simulation environments are often evaluated on CPUs, not GPUs. If Kaggle evaluates this on CPU, the LLM may take too long and cause a Timeout Error. Be prepared to switch to a highly compressed model or heuristic engine if that happens.

In [1]:
%%writefile submission.py
import json
import re
import torch
import os
import sys
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# 1. LOAD MODEL GLOBALLY (Runs once on boot)
# ==========================================
# CRITICAL: Change this path to your uploaded Kaggle Dataset path!
# Example: MODEL_PATH = "/kaggle/input/qwen2-1-5b-instruct/transformers/default/1"
MODEL_PATH = "Qwen/Qwen2-1.5B-Instruct" # Use internet-hosted for local testing only

# Determine if GPU is available in the Kaggle Evaluation Backend
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map=device,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

# ==========================================
# 2. OBSERVATION PARSER
# ==========================================
def format_game_state_to_prompt(obs):
    try:
        my_active = obs.get('my_active', 'Unknown Pokemon')
        my_hand = obs.get('my_hand', [])
        opp_active = obs.get('opp_active', 'Unknown Pokemon')
        valid_actions = obs.get('valid_actions', [])
        
        prompt = (
            "You are an expert Pokemon TCG player. Evaluate the game state and pick the best action index.\n"
            f"Your Active Pokemon: {my_active}\n"
            f"Your Hand: {', '.join([str(c) for c in my_hand])}\n"
            f"Opponent Active: {opp_active}\n"
            "Valid Actions:\n"
        )
        for i, action in enumerate(valid_actions):
            prompt += f"[{i}] {action}\n"
        prompt += "\nBased on an Aggro strategy, output ONLY the single integer index inside brackets, like [0]."
        return prompt, valid_actions
    except Exception as e:
        return str(obs), []

# ==========================================
# 3. THE KAGGLE AGENT
# ==========================================
def agent(obs, conf):
    prompt, valid_actions = format_game_state_to_prompt(obs)
    
    if len(valid_actions) <= 1:
        return 0

    messages = [
        {"role": "system", "content": "You are a Pokemon TCG AI. Return only the integer inside brackets, e.g., [1]."},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    outputs = model.generate(**inputs, max_new_tokens=5, temperature=0.1, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    try:
        match = re.search(r'\[(\d+)\]', response)
        if match:
            action_idx = int(match.group(1))
            if 0 <= action_idx < len(valid_actions):
                return action_idx
    except:
        pass
        
    return 0


Writing submission.py
